# Session 2: Interpretable Autoencoders
## Building a Simple Autoencoder for Chest X-rays

In this notebook, we're going to take our first step into GenAI for Life Science. We'll be working with the **PneumoniaMNIST** dataset - these are 28x28 pixel images of chest X-rays, each labelled as either *Normal* or *Pneumonia*. Our goal in this session is to teach a small **autoencoder** to compress every image down to just **two numbers** and then rebuild it.

Two numbers may sound like a ridiculously small budget for a chest X-ray, but that forces our model to find a way to summarise what really matters in the image.

Working through this notebook, we're going to:

1. Set up the environment.
2. Download the dataset.
3. Train a simple autoencoder.
4. Investigate the test latent space.
5. Build an MLP to classify images from the latent space.
6. Feed the decoder made-up latent vectors and see what it generates.

Run the notebook one cell at a time and complete the activities to the best of your ability.

## 1. Setup

In this cell, we're going to load our libraries and fix a random seed. We'll use many of the same libraries as in Session 1, with the addition of:
- sklearn - Scikit-learn, which gives us ready-made tools for scoring a classifier. We'll use this to plot a confusion matrix later.
- medmnist — The MedMNIST library, which downloads standardised biomedical image datasets including PneumoniaMNIST. We also import the INFO dictionary, which contains metadata (class labels, dataset names, image shapes).

If you haven't already, make sure to put the downloaded zip folders from the bottom of **Setup** into this directory.

In [ ]:
# See Session 1 for a description of these libraries
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import medmnist
from medmnist import INFO

# Fix a random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set the device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Download PneumoniaMNIST

[MedMNIST](https://medmnist.com/) is a collection of standardised biomedical image datasets. One of those datasets is **PneumoniaMNIST**, which contains 5,856 chest X-rays (28x28 pixel, grayscale images) labelled as either *Normal* (0) or *Pneumonia* (1).

Remember that an image can be represented as a matrix, where each entry is a pixel, and the value of that entry represents the pixel's brightness or colour. This means we have a set of images (matrices/tensors of size 28 by 28) and a vector of 0s and 1s, corresponding to the class of each image.

You should have downloaded the data already (see Setup in the notes), but if you wanted to be clever and download it directly you could use the cell below. If you wanted to use a different MedMNIST dataset (e.g. BloodMNIST, OrganAMNIST), just change the `DATA_FLAG`. Keep in mind that this notebook is only compatible with **PneumoniaMNIST**.

**Activity:**
- Print the number of images in the training, validation and test set using the `len` function.
- You can access the training images directly through `train_ds.imgs` Print the shape of one of these images using the `.shape` property.

In [ ]:
# Download PneumoniaMNIST
# This is the identifier for the PneumoniaMNIST dataset
DATA_FLAG = "pneumoniamnist"
# This is where we'll save the data (so we don't have to download it every time)
DATA_DIR = os.path.abspath("./data_session1-2")
# Make our data directory (create a new folder) if it doesn't exist
os.makedirs(DATA_DIR, exist_ok=True)

# medmnist gives us a metadata dict per dataset - this contains the class labels of each image
# Here, we pull the PneumoniaMNIST metadata specifically
info = INFO[DATA_FLAG]
# Get the actual Python class for this dataset, this basically tells Python we want to use the PneumoniaMNIST data
DataClass = getattr(medmnist, info["python_class"])
# Count how many classes the dataset has
N_CLASSES = len(info["label"])
print(f"Dataset: {info['python_class']}")
print(f"Number of classes: {N_CLASSES}  ->  {info['label']}")

# MedMNIST already gives us a train / val / test split - we'll use it as-is
# Here, we download the training, validation and test splits by setting download=True and root=DATA_DIR
train_ds = DataClass(split="train", download=True, root=DATA_DIR)
val_ds = DataClass(split="val",   download=True, root=DATA_DIR)
test_ds = DataClass(split="test",  download=True, root=DATA_DIR)

### Convert to Tensors
In order to make our image data compatible with PyTorch, we need to convert them into PyTorch tensors. We'll normalise the pixel values to between 0 and 1 (this improves training stability). We also add a channel dimension so each image has shape (1, 28, 28) - this is what PyTorch expects.

**Activity:** Once you've worked through the notebook, try changing the batch size in the cell below. See how this effects training speed and stability.

In [ ]:
# We define a helper function to convert our data into tensors
def to_tensors(ds):
    # By default, each pixel is an 8-bit integer (0-255), so we need to convert to a float and normalise to [0,1].
    # Here, .float() converts to a float, .unsqueeze(1) adds a channel dimension, and / 255.0 normalises to [0,1].
    x = torch.from_numpy(ds.imgs).float().unsqueeze(1) / 255.0
    # We also need to convert the labels to a tensor, we use .squeeze(-1) to re-format the labels into a vector
    y = torch.from_numpy(ds.labels.astype(np.int64)).squeeze(-1)
    # Return the image tensor and label tensor
    return x, y

# Now we can run this function on our data and convert the training, validation and test splits to tensors
x_train, y_train = to_tensors(train_ds)
x_val,   y_val   = to_tensors(val_ds)
x_test,  y_test  = to_tensors(test_ds)

# Here, we set our batch size - this is the number of images we will put through the model at a time.
# Try changing the batch size and see how it affects training.
BATCH_SIZE = 128
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(x_val,   y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(TensorDataset(x_test,  y_test),  batch_size=BATCH_SIZE)

### Plot the Data
Always look at your data before you train anything - here are six examples of each class side by side. Can you notice any differences?

In [ ]:
# This block of code plots a 2 by 6 grid of images (i.e. six images from each class).
fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
# We have two classes, Normal and Pneumonia, so we label them accordingly.
class_names = ["Normal", "Pneumonia"]
# Iterate over each class - row 0 for Normal, row 1 for Pneumonia
for row, cls in enumerate([0, 1]):
    # Indices of the first six training images belonging to this class
    idxs = (y_train == cls).nonzero(as_tuple=True)[0][:6]
    # Loop over those six example indices
    for col, idx in enumerate(idxs):
        # Show the image in greyscale (channel dim 0 is the only channel)
        axes[row, col].imshow(x_train[idx, 0], cmap="gray")
        # Label each subplot with its class name
        axes[row, col].set_title(class_names[cls], fontsize=9)
        # Hide tick marks and axis lines for a cleaner figure
        axes[row, col].axis("off")
# Overall title for the figure
plt.suptitle("PneumoniaMNIST samples")
# Adjust subplot spacing so labels don't overlap
plt.tight_layout()
# Render the figure
plt.show()

# Can you see the difference?
# Pneumonia typically presents in x-rays as white cloudy patches where the lungs should normally be dark.

## 3. A simple autoencoder

In this cell, we're going to build a very simple autoencoder. Remember, an autoencoder is designed to compress our data (a big grid of numbers) into a much smaller set of numbers, which we call a **latent representation**. We do this by using an encoder (which gradually compresses our data) and a decoder (a mirror of the encoder which tries to reconstruct the original image).

We evaluate model performance by calculating the difference between our input image and the model's output - one of the simplest ways to do this is through **mean squared error** (MSE), which works out the average squared difference between each pixel in the original image and the reconstruction. We want our MSE to be as small as possible - if the two images are perfectly identical, the MSE will be 0. So, we set MSE as our loss and try to update our model parameters to minimise it.

In this case, our architecture contains just one fully-connected encoder block, one fully-connected decoder block, and a **2D** latent space in the middle (just two numbers).

**Activity:** Try adding more layers to the autoencoder. See how this effects the total number of trainable parameters.

In [ ]:
# Here's where we define the autoencoder using PyTorch
# In this case, we define the autoencoder as a class, which allows us to call relevant methods (e.g. encode and decode)
# separately. The principles are still the same as in Session 1.
# We use the subclass nn.Module to let Python know we're defining a new neural network
class SimpleAutoencoder(nn.Module):
    """One-block fully-connected autoencoder with a 2D latent space."""

    # These scary lines of code basically just set some important default numbers the network needs to remember,
    # e.g. the size of the input image, the size of the hidden layer, and the size of the latent space.
    def __init__(self, input_dim: int = 28 * 28, hidden_dim: int = 128, latent_dim: int = 2):
        super().__init__()
        # Store the size of our input image. Our network requires a vector input, so instead of passing an entire
        # 28 by 28 pixel image, we pass a flattened vector of 28*28 pixels.
        self.input_dim = input_dim
        # Here we define the encoder layers
        self.encoder = nn.Sequential(
            # Project the flattened image down to the hidden width
            nn.Linear(input_dim, hidden_dim),
            # Use a ReLU activation function between layers (we set inplace=True to save memory)
            nn.ReLU(inplace=True),
            # Project hidden representation down to the 2D latent space
            nn.Linear(hidden_dim, latent_dim),
        )
        # And here are the decoder layers
        self.decoder = nn.Sequential(
            # Expand the latent vector back up to the hidden width
            nn.Linear(latent_dim, hidden_dim),
            # Use a ReLU activation function between layers
            nn.ReLU(inplace=True),
            # Project hidden representation back up to pixel space (original image size)
            nn.Linear(hidden_dim, input_dim),
            # Use a sigmoid activation function to map pixels back to [0, 1] range
            nn.Sigmoid(),
        )

    # This function encodes the input image into a 2D latent space by calling the encoder layers
    def encode(self, x):
        # Flatten image batch from (N,1,28,28) to (N,784), then run through the encoder
        return self.encoder(x.view(x.size(0), -1))

    # This function decodes the latent space back into an image by calling the decoder layers
    def decode(self, z):
        # Run the latent through the decoder and reshape back to image dimensions
        return self.decoder(z).view(-1, 1, 28, 28)

    # This function puts the encoder and decoder together and returns the encoded image and the latent space
    def forward(self, x):
        # Encode the input batch into a latent vector (here, z represents two numbers)
        z = self.encode(x)
        # Decode the latent vector back into a reconstructed image (x_hat is our reconstruction)
        x_hat = self.decode(z)
        # Return both reconstruction and latent representation
        return x_hat, z

# Let's put it all together
# Set up the autoencoder and move it onto the chosen device
simple_ae = SimpleAutoencoder().to(DEVICE)
# Print the architecture so we can sanity-check the layer stack
print(simple_ae)
# Count total trainable parameters (sum of numel() over every learnable tensor)
n_params = sum(p.numel() for p in simple_ae.parameters())
print(f"Trainable parameters: {n_params:,}")
# 202,258 parameters is actually pretty low - most models you're familiar with will have millions if not billions

### Training loop

Now we'll train our autoencoder. Model training follows the same steps for each epoch:
1. Set model to training mode.
2. Pass data through the model (usually in batches).
3. Determine our output (in this case, an image reconstruction).
4. Calculate loss (here, MSE between the input and reconstruction).
5. Reset ("zero out") gradients from the previous step.
6. Backpropagate to compute the gradient of the loss with respect to the model parameters.
7. Use the optimiser to calculate parameter updates.
8. Repeat until we reach the target number of epochs.

We'll start by using 10 epochs and the Adam optimiser with a learning rate of 0.001.

**Activity:** Try changing the number of epochs and learning rate in the training cell below. See how this effects the final training and validation MSE.

In [ ]:
# This time, we define a function for training the autoencoder, so we can easily reuse it later.
# We input the model, training and validation dataloaders, number of epochs and learning rate for the optimiser
def train_autoencoder(model, train_loader, val_loader, epochs=10, lr=0.001):
    # Set the optimiser (Adam)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    # Set up a dictionary to record train and validation loss at every epoch
    history = {"train_mse": [], "val_mse": []}
    # Loop over epochs (one epoch = one full pass over the training set)
    for epoch in range(1, epochs + 1):
        # Put the model in training mode
        model.train()
        # Set a running sum of the training loss across batches
        running = 0.0
        # Iterate over training minibatches - the train_loader gives us a pair of image batches and their labels.
        # For the autoencoder, we only need the images (xb) so we ignore the labels (_)
        for xb, _ in train_loader:
            # Move the batch onto the chosen device
            xb = xb.to(DEVICE)
            # Forward pass - put the batch through the model and get the reconstructions
            # Remember, the model outputs both the reconstruction (x_hat) and the latent vector
            # (we ignore the latent vector with _)
            x_hat, _ = model(xb)
            # Calculate mean squared error loss between reconstruction and input
            loss = F.mse_loss(x_hat, xb)
            # Reset gradients from the previous step
            opt.zero_grad()
            # Backpropagate the loss to compute gradient of loss with respect to model parameters
            loss.backward()
            # Update model parameters using optimiser
            opt.step()
            # Record the total loss so far (for averaging later). We multiply by xb.size(0) because the
            # loss is a batch average.
            running += loss.item() * xb.size(0)
        # Once we have passed all batches, calculate the mean training loss for the epoch
        train_mse = running / len(train_loader.dataset)

        # Switch to evaluation mode
        model.eval()
        # Reset the running loss accumulator for validation
        running = 0.0
        # Disable autograd for validation (we don't need backpropagation here)
        with torch.no_grad():
            # Iterate over validation batches
            for xb, _ in val_loader:
                # Move the batch onto the chosen device
                xb = xb.to(DEVICE)
                # Forward pass on validation data
                x_hat, _ = model(xb)
                # Record the total loss so far (for averaging later). Here we can use reduction="sum" to
                # automatically sum the loss across all pixels and all batches, then divide by the total
                # number of pixels per image.
                running += F.mse_loss(x_hat, xb, reduction="sum").item() / xb[0].numel()
        # Mean validation MSE for the epoch
        val_mse = running / len(val_loader.dataset)

        # Record this epoch's training and validation losses
        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)
        print(f"Epoch {epoch:02d} | train MSE {train_mse:.4f} | val MSE {val_mse:.4f}")
    # Return the loss history so we can plot it later
    return history

# Now that we've defined the training loop, we can run training from a single line.
history_simple = train_autoencoder(simple_ae, train_loader, val_loader, epochs=10, lr=1e-3)


## 4. Evaluate the model on the test set

Now let's see how our model performs on the held-out test set. We're going to do the following:
* Pass our test images through the trained model. This is largely the same as what we did for the validation data above.
* Record the mean squared error across all images in the test set.
* Plot the latent space, colouring each point by its true class label. Remember, the model was never told the labels, so any class separation is purely emergent behaviour.
* Look at a few reconstructions next to the originals.

In [ ]:
# Again, we disable autograd because we're evaluating the model, not training it.
@torch.no_grad()
# Define a function to evaluate the model, which takes the model and the test dataloader as inputs.
def evaluate(model, loader):
    # Put the model in evaluation mode.
    model.eval()
    # Set a running sum of the loss across batches
    total = 0.0
    # Define some empty lists to gather latent vectors, labels, reconstructions and originals across batches
    zs, ys, recons, originals = [], [], [], []
    # Iterate over the dataloader, unpacking both images and labels this time (for plotting later)
    for xb, yb in loader:
        # Move the input batch onto the chosen device
        xb = xb.to(DEVICE)
        # Forward pass - get both the reconstruction and the latent vector
        x_hat, z = model(xb)
        # Calculate loss
        total += F.mse_loss(x_hat, xb, reduction="sum").item() / xb[0].numel()
        # Store the latent vectors
        zs.append(z.cpu().numpy())
        # Store the labels
        ys.append(yb.numpy())
        # Store the reconstructions
        recons.append(x_hat.cpu().numpy())
        # And store the originals
        originals.append(xb.cpu().numpy())
    # Calculate the mean MSE across the whole dataset
    mse = total / len(loader.dataset)
    # Return MSE plus all the collected arrays
    return mse, np.concatenate(zs), np.concatenate(ys), np.concatenate(recons), np.concatenate(originals)

# Define a helper function to plot the latent space
def plot_latent(z, y, title):
    # Create a single figure
    fig, ax = plt.subplots(figsize=(5.5, 5))
    # Map class labels to two distinct colours (red and blue here)
    palette = {0: "#3b82f6", 1: "#ef4444"}
    # Plot each class as its own scatter plot
    for cls, name in [(0, "Normal"), (1, "Pneumonia")]:
        # Boolean mask selecting the rows for this class
        mask = y == cls
        # Scatter the latents of this class in the chosen colour
        ax.scatter(z[mask, 0], z[mask, 1], s=8, alpha=0.6, c=palette[cls], label=name)
    # Label the x-axis
    ax.set_xlabel("latent dim 1")
    # Label the y-axis
    ax.set_ylabel("latent dim 2")
    # Set the plot title
    ax.set_title(title, fontsize=16)
    # Show the per-class legend
    ax.legend()
    # Tidy up subplot spacing
    plt.tight_layout()
    # Show the figure
    plt.show()

# Define function to plot reconstructions
def plot_reconstructions(orig, recon, n=8, title=""):
    # Create a 2-row grid: row 0 for originals, row 1 for reconstructions
    fig, axes = plt.subplots(2, n, figsize=(1.4 * n, 3.0))
    # Loop over the first n examples
    for i in range(n):
        # Show the original image in the top row
        axes[0, i].imshow(orig[i, 0], cmap="gray")
        # Show the reconstruction in the bottom row
        axes[1, i].imshow(recon[i, 0], cmap="gray")
        # Hide ticks on both panels, but keep the axes so we can label the rows
        for row in (0, 1):
            axes[row, i].set_xticks([])
            axes[row, i].set_yticks([])
    # Label the originals row
    axes[0, 0].set_ylabel("Original", fontsize=10)
    # Label the reconstructions row
    axes[1, 0].set_ylabel("Reconstruction", fontsize=10)
    # Overall figure title
    plt.suptitle(title, fontsize=16)
    # Tidy up subplot spacing
    plt.tight_layout()
    # Show the figure
    plt.show()

# Get the model outputs and plot them
mse_simple, z_simple, y_simple, recon_simple, orig_simple = evaluate(simple_ae, test_loader)
print(f"Test MSE: {mse_simple:.4f}")
# Plot the latent space coloured by class
plot_latent(z_simple, y_simple, "Test latent space")
# Plot a few test reconstructions next to their originals
plot_reconstructions(orig_simple, recon_simple, title="Reconstructions")


## 5. Classifying directly from the latent space

The autoencoder was never shown the class labels, yet the latent space above already looks partially separated. Let's test that properly by training a small classifier that only ever sees the two latent numbers.

Our classifier is a small MLP with one hidden layer and a little dropout. It outputs a **logit** for each class, which is just a score for how much the latent vector seems to belong to that class. Applying a **softmax** turns those scores into probabilities, and we take the highest probability as the prediction. We train it with cross-entropy loss on the latent representations, then measure accuracy on the held-out test set.

**Activity:** Try changing the hidden width, the dropout rate, the learning rate or the number of epochs. How much accuracy can you get out of just two numbers per image?

In [ ]:
# First, we define a helper function which encodes a whole dataset into latent vectors.
@torch.no_grad()
def get_latents(model, loader):
    # Put the model in evaluation mode
    model.eval()
    # Define some empty lists to gather latent vectors and labels across batches
    zs, ys = [], []
    # Iterate over the dataloader, unpacking both images and labels
    for xb, yb in loader:
        # Move the batch onto the chosen device, then encode it into the latent space
        z = model.encode(xb.to(DEVICE))
        # Store the latent vectors
        zs.append(z)
        # Store the matching labels
        ys.append(yb)
    # Stack the results into one tensor of latents and one tensor of labels
    return torch.cat(zs), torch.cat(ys)

# Here, we create a small MLP that classifies latent vectors as "Normal" or "Pneumonia"
class LatentMLP(nn.Module):
    # Define the important variables (latent dimension, hidden dimension, number of classes)
    def __init__(self, latent_dim, hidden=32, n_classes=2):
        super().__init__()
        # This is a simple two-layer MLP with dropout
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden, n_classes),
        )

    # Forward pass returns the logits for each class
    def forward(self, z):
        return self.net(z)

# Encode the training and test images into the latent space
z_train, labels_train = get_latents(simple_ae, train_loader)
z_test, labels_test = get_latents(simple_ae, test_loader)

# Wrap the training latents in a DataLoader
latent_loader = DataLoader(TensorDataset(z_train, labels_train), batch_size=BATCH_SIZE, shuffle=True)

# Build the classifier
clf = LatentMLP(latent_dim=z_train.size(1)).to(DEVICE)
# Set the optimiser
clf_opt = torch.optim.Adam(clf.parameters(), lr=0.01)
# Number of epochs
CLF_EPOCHS = 20

# The training loop follows the same steps as the autoencoder's, only the loss has changed
for epoch in range(1, CLF_EPOCHS + 1):
    # Put the classifier in training mode (this switches dropout on)
    clf.train()
    # Set a running sum of the training loss across batches
    running = 0.0
    # Iterate over minibatches of latent vectors and their labels
    for zb, yb in latent_loader:
        # Move the batch onto the chosen device
        zb, yb = zb.to(DEVICE), yb.to(DEVICE)
        # Forward pass
        logits = clf(zb)
        # Cross-entropy loss
        loss = F.cross_entropy(logits, yb)
        # Reset gradients from the previous step
        clf_opt.zero_grad()
        # Backpropagate
        loss.backward()
        # Update model parameters
        clf_opt.step()
        # Record the total loss so far
        running += loss.item() * zb.size(0)
    # Mean training loss for this epoch
    train_loss = running / len(latent_loader.dataset)
    # Print progress every fifth epoch
    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | train loss {train_loss:.4f}")

# Let's see how the classifier does on the held-out test set
# Switch to evaluation mode (this switches dropout off)
clf.eval()
# Disable autograd
with torch.no_grad():
    # Get the logits for every test latent vector
    test_logits = clf(z_test.to(DEVICE))
    # Convert those logits into a probability for each class
    test_probs = F.softmax(test_logits, dim=1)
    # The prediction is the class with the highest probability
    test_pred = test_probs.argmax(dim=1).cpu()

# Calculate accuracy as the proportion of test images we classified correctly
clf_acc = (test_pred == labels_test).float().mean().item()
print(f"\nTest accuracy from two latent numbers alone: {clf_acc:.3f}")

### Plotting the confusion matrix

In a **confusion matrix**, each row is a true class and each column is a predicted class, so the diagonal holds the correct predictions and everything off-diagonal is a mistake. Here we normalise each row, meaning the numbers show the proportion of each true class that was assigned to each prediction.

In [ ]:
# Calculate the confusion matrix, normalising each row so it sums to 1
cm = confusion_matrix(labels_test.numpy(), test_pred.numpy(), labels=[0, 1], normalize="true")

# Create the figure
fig, ax = plt.subplots(figsize=(5.5, 4.5))
# Draw the matrix
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, colorbar=False, values_format=".2f")
# Title showing the overall accuracy
ax.set_title(f"MLP (acc={clf_acc:.2f})", fontsize=14)
# Tidy up subplot spacing
plt.tight_layout()
# Show the figure
plt.show()

## 6. Generate synthetic X-rays from fake latent vectors

Now we can turn our autoencoder into a (very simple) generative model. Once it's trained, the decoder is just a function from a 2D vector to a 28x28 image - so if we make up a 2D vector, the decoder will happily draw us an X-ray for it. It doesn't memorise individual training images, so the picture it produces is genuinely synthetic.

We're going to do two things:

1. Decode a handful of arbitrary latent vectors so we can see what specific points in the plane look like.
2. Sweep a regular grid across a chunk of the latent plane and tile the resulting images into a single panel. This is a nice way to see how the decoder's imagined X-rays change as you move around the latent space.

In [ ]:
# Disable autograd for the whole function (we're only doing inference)
@torch.no_grad()
# Define a function to apply the decoder to the latent vector
def decode_latents(model, z):
    # Convert the input to a float tensor on the chosen device
    z = torch.as_tensor(z, dtype=torch.float32, device=DEVICE)
    # Decode the latent and return the result as a NumPy array
    return model.decode(z).cpu().numpy()


# First, let's just decode a handful of arbitrary, hand-picked latent points
z_query = np.array([
    [-2.0, -2.0],
    [ 0.0,  0.0],
    [ 2.0,  2.0],
    [-2.0,  2.0],
    [ 2.0, -2.0],
], dtype=np.float32)
# Here we get our images
samples = decode_latents(simple_ae, z_query)
# And then we plot them
fig, axes = plt.subplots(1, len(z_query), figsize=(2.0 * len(z_query), 2.4))
# Iterate over each pair
for i, (ax, z) in enumerate(zip(axes, z_query)):
    # Show the decoded image and hide axes
    ax.imshow(samples[i, 0], cmap="gray"); ax.axis("off")
    # Title shows the latent coordinates that generated this image
    ax.set_title(f"z=({z[0]:+.1f},{z[1]:+.1f})", fontsize=9)
# Overall figure title
plt.suptitle("Synthetic X-rays from hand-picked latent vectors")
# Tidy spacing and render
plt.tight_layout(); plt.show()

In [ ]:
# Now, let's sample uniformly across the latent space and tile the decoded images together.
# Number of grid points along each latent axis
grid_n = 10
# Range of the first latent dimension
xlim = (-2.0, 2.0)
# Range of the second latent dimension
ylim = (-2.0, 2.0)
# Evenly spaced sample points along dim 1
gx = np.linspace(xlim[0], xlim[1], grid_n)
# Evenly spaced sample points along dim 2
gy = np.linspace(ylim[0], ylim[1], grid_n)
# Build the grid and flatten it to (N, 2) for input into the decoder
zz = np.stack(np.meshgrid(gx, gy), axis=-1).reshape(-1, 2)
# Decode every grid point and reshape into a (grid_n, grid_n) panel of 28x28 images
imgs = decode_latents(simple_ae, zz).reshape(grid_n, grid_n, 28, 28)
# Create a figure to hold the tiled image
fig, ax = plt.subplots(figsize=(7, 7))
# Blank canvas big enough to hold every tile
canvas = np.zeros((28 * grid_n, 28 * grid_n))
# Drop each decoded image into the correct spot on the canvas
for i in range(grid_n):
    for j in range(grid_n):
        # Flip i so that the lowest y is at the bottom of the figure
        canvas[(grid_n - 1 - i) * 28:(grid_n - i) * 28, j * 28:(j + 1) * 28] = imgs[i, j]
# Display the assembled canvas with axes annotated by latent coordinates
ax.imshow(canvas, cmap="gray", extent=[xlim[0], xlim[1], ylim[0], ylim[1]], aspect="auto")
# Latent-axis labels
ax.set_xlabel("latent dim 1"); ax.set_ylabel("latent dim 2")
# Figure title
ax.set_title("Decoder sweep over the 2D latent plane")
# Tidy and show
plt.tight_layout(); plt.show()

# Some of these images look realistic, but others look like nonsense. That's because we've sampled from quite far out in the latent space,
# where the decoder has never been trained. As such, the output becomes unpredictable.

### Picking the sweep limits automatically

Let's try picking latent values in a more data-driven manner. We'll centre the grid on the mean latent vector of the test set and extend $\pm 2\sigma$ along each axis (where $\sigma$ is the standard deviation). If we assume our latent space is roughly Gaussian, this should cover 90% of the probable latent vectors.

We'll plot two figures: the clean tile of decoded images first, and then the same tile with the real test latents scattered on top so we can check that the sweep really does match the data.

In [ ]:
# Get the per-dimension mean of the test latents
mu  = z_simple.mean(axis=0)
# Get the per-dimension standard deviation of the test latents
sig = z_simple.std(axis=0)
# Set up x and y limits for the grid
xlim_auto = (float(mu[0] - 2 * sig[0]), float(mu[0] + 2 * sig[0]))
ylim_auto = (float(mu[1] - 2 * sig[1]), float(mu[1] + 2 * sig[1]))

# Grid resolution along each latent axis
grid_n = 10
# Evenly spaced sample points along dim 1
gx = np.linspace(xlim_auto[0], xlim_auto[1], grid_n)
# Evenly spaced sample points along dim 2
gy = np.linspace(ylim_auto[0], ylim_auto[1], grid_n)
# Build the grid and flatten it to (N, 2) for input into the decoder
zz = np.stack(np.meshgrid(gx, gy), axis=-1).reshape(-1, 2)
# Decode every grid point and reshape into a panel of 28x28 images
imgs = decode_latents(simple_ae, zz).reshape(grid_n, grid_n, 28, 28)

# Plot the decoded images
canvas = np.zeros((28 * grid_n, 28 * grid_n))
# Iterate over every grid cell
for i in range(grid_n):
    for j in range(grid_n):
        # Drop each decoded image into the right tile (flipping i so y increases upward)
        canvas[(grid_n - 1 - i) * 28:(grid_n - i) * 28, j * 28:(j + 1) * 28] = imgs[i, j]

# Create figure for the clean tiled image
fig, ax = plt.subplots(figsize=(7, 7))
# Display the canvas with axes annotated by the actual latent ranges
ax.imshow(
    canvas, cmap="gray",
    extent=[xlim_auto[0], xlim_auto[1], ylim_auto[0], ylim_auto[1]],
    aspect="auto",
)
# Axis labels
ax.set_xlabel("latent dim 1"); ax.set_ylabel("latent dim 2")
# Figure title
ax.set_title(r"Decoder sweep over mean $\pm 2\sigma$")
# Tidy and show
plt.tight_layout(); plt.show()

# Second figure: same canvas but with the real test latents overlaid for comparison
fig, ax = plt.subplots(figsize=(7, 7))
# Re-display the canvas
ax.imshow(
    canvas, cmap="gray",
    extent=[xlim_auto[0], xlim_auto[1], ylim_auto[0], ylim_auto[1]],
    aspect="auto",
)
# Now overlay the real test latents on top, coloured by class label
ax.scatter(
    z_simple[:, 0], z_simple[:, 1],
    c=y_simple, cmap="coolwarm", s=6, alpha=0.5, edgecolors="none",
)
# Axis labels
ax.set_xlabel("latent dim 1"); ax.set_ylabel("latent dim 2")
# Figure title
ax.set_title(r"Same sweep with test latents overlaid")
# Tidy and show
plt.tight_layout(); plt.show()

# That's better - remember, those images you're seeing are not real lungs, they're completely AI-generated
# What can we see from the latent space? Seems like the cloudy, hazy images are more likely to be pneumonia.

## Final Activity
If you've made it this far, go back through the notebook and see how low you can push the test accuracy. Try changing the batch size, number of epochs, learning rate and model architecture.